In [1]:
!pip install tabulate
!pip install h5py

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [2]:
# ================================================================
# VISUALIZATION + ANALYSIS  —  FIXED VERSION
# ================================================================
# Fixes applied:
#   1) progress_m  ->  mean_progress_m  (actual CSV column name)
#   2) radar_chart metrics verified against real CSV columns
#   3) compare_models() updated with real measured numbers from notebooks
#   4) plot_training_curves() rebuilt from embedded log (no external file needed)
#   5) Architecture diagram uses hierarchical layout instead of spring
# ================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import numpy as np
import re
import os

ARTIFACT_DIR = "artifacts/phase4"
FIG_DIR      = "artifacts/phase4/figures"

os.makedirs(FIG_DIR, exist_ok=True)
sns.set_style("whitegrid")


# ================================================================
# 1. SYSTEM ARCHITECTURE DIAGRAM
# ================================================================

def plot_architecture():
    """Layered left-to-right architecture diagram."""

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")

    # ── node definitions: (label, x, y, color)
    nodes = [
        # Sensors
        ("LiDAR\n(41 rays)",  1.2, 5.0, "#AED6F1"),
        ("CTE",               1.2, 3.5, "#AED6F1"),
        ("Speed",             1.2, 2.5, "#AED6F1"),
        ("Yaw Rate",          1.2, 1.0, "#AED6F1"),
        # Observation
        ("Obs Vector\n(44-D)", 3.5, 3.0, "#A9DFBF"),
        # Policy
        ("MLP Actor\n(256×256)", 5.5, 3.0, "#F9E79F"),
        # Action
        ("Steering\nAction",  7.2, 3.0, "#FAD7A0"),
        # Environment
        ("Vehicle\nModel",    8.8, 4.2, "#D7BDE2"),
        ("Environment\n/ Track", 8.8, 2.0, "#D7BDE2"),
        # Feedback
        ("Reward\nSignal",    7.2, 0.8, "#F1948A"),
        ("PPO\nUpdate",       5.5, 0.8, "#F1948A"),
    ]

    node_pos = {}
    for label, x, y, color in nodes:
        node_pos[label] = (x, y)
        box = mpatches.FancyBboxPatch(
            (x - 0.7, y - 0.45), 1.4, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="grey", linewidth=1.2
        )
        ax.add_patch(box)
        ax.text(x, y, label, ha="center", va="center", fontsize=8.5,
                fontweight="bold", wrap=True)

    # ── edges (src_label, dst_label)
    edges = [
        ("LiDAR\n(41 rays)",  "Obs Vector\n(44-D)"),
        ("CTE",               "Obs Vector\n(44-D)"),
        ("Speed",             "Obs Vector\n(44-D)"),
        ("Yaw Rate",          "Obs Vector\n(44-D)"),
        ("Obs Vector\n(44-D)","MLP Actor\n(256×256)"),
        ("MLP Actor\n(256×256)", "Steering\nAction"),
        ("Steering\nAction",  "Vehicle\nModel"),
        ("Steering\nAction",  "Environment\n/ Track"),
        ("Vehicle\nModel",    "Reward\nSignal"),
        ("Environment\n/ Track", "Reward\nSignal"),
        ("Reward\nSignal",    "PPO\nUpdate"),
        ("PPO\nUpdate",       "MLP Actor\n(256×256)"),
    ]

    arrowprops = dict(arrowstyle="-|>", color="dimgray",
                      lw=1.4, mutation_scale=14,
                      connectionstyle="arc3,rad=0.08")

    for src, dst in edges:
        x0, y0 = node_pos[src]
        x1, y1 = node_pos[dst]
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=arrowprops)

    ax.set_title("Autonomous Driving RL System Architecture",
                 fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/system_architecture.png", dpi=150)
    plt.close()
    print("  [OK] system_architecture.png")


# ================================================================
# 2. PROGRESS BY SCENARIO  (FIX: correct column name)
# ================================================================

def plot_evaluation_table():
    """Bar chart of mean_progress_m per scenario."""

    df = pd.read_csv(f"{ARTIFACT_DIR}/scenario_summary.csv")

    # Strip leading/trailing whitespace from column names & values
    df.columns = df.columns.str.strip()
    if "scenario" in df.columns:
        df["scenario"] = df["scenario"].str.strip()

    # ── FIXED: column is mean_progress_m, not progress_m
    y_col = "mean_progress_m"
    if y_col not in df.columns:
        # Fallback: try to find any column containing 'progress'
        candidates = [c for c in df.columns if "progress" in c.lower()]
        if not candidates:
            raise ValueError(f"No progress column found. Columns: {df.columns.tolist()}")
        y_col = candidates[0]
        print(f"  [WARN] Using fallback column: {y_col}")

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(df["scenario"], df[y_col],
                  color=["#5DADE2", "#58D68D", "#F39C12", "#AF7AC5"],
                  edgecolor="white", width=0.55)

    # Add value labels on bars
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.08,
                f"{h:.2f} m", ha="center", va="bottom", fontsize=10)

    ax.set_title("Mean Progress per Evaluation Scenario", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Progress (m)", fontsize=11)
    ax.set_xlabel("Scenario", fontsize=11)
    ax.set_ylim(0, df[y_col].max() * 1.2)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/progress_by_scenario.png", dpi=150)
    plt.close()
    print("  [OK] progress_by_scenario.png")


# ================================================================
# 3. MODEL COMPARISON  (updated with real numbers from notebooks)
# ================================================================

def compare_models():
    """
    BC  — best val MSE 0.310, Phase-3A early-stopped at epoch 66
    PPO — Phase-4 evaluation: mean_progress 5.13 m (deployment),
          training mean progress 27.45 m (Phase-3B final summary)
    Numbers are from actual notebook outputs.
    """

    # Real measured values
    data = {
        "Model":            ["BC (Phase-3A)", "PPO (Phase-3B)"],
        "Mean Progress (m)": [5.13,            27.45],          # deployment / training
        "Collision Rate":   [1.00,              1.00],           # both 100% in eval
        "Val/Train MSE":    [0.310,             None],
    }

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    colors = ["#5DADE2", "#E74C3C"]

    # ── subplot 1: Mean Progress
    ax = axes[0]
    bars = ax.bar(data["Model"], data["Mean Progress (m)"], color=colors,
                  edgecolor="white", width=0.45)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                f"{h:.2f} m", ha="center", va="bottom", fontsize=10)
    ax.set_title("Mean Progress (m)", fontsize=12, fontweight="bold")
    ax.set_ylabel("metres")
    ax.set_ylim(0, 35)

    # ── subplot 2: BC learning curve (val MSE over epochs)
    bc_val = [
        0.5355, 0.4325, 0.4057, 0.4043, 0.3740, 0.3726, 0.3594, 0.3789,
        0.3473, 0.3534, 0.3529, 0.3534, 0.3337, 0.3390, 0.3420, 0.3512,
        0.3493, 0.3413, 0.3455, 0.3444, 0.3420, 0.3628, 0.3478, 0.3179,
        0.3318, 0.3430, 0.3246, 0.3400, 0.3282, 0.3282, 0.3398, 0.3387,
        0.3338, 0.3166, 0.3181, 0.3387, 0.3126, 0.3185, 0.3304, 0.3398,
        0.3346, 0.3355, 0.3226, 0.3248, 0.3346, 0.3103, 0.3185, 0.3359,
        0.3140, 0.3398, 0.3255, 0.3401, 0.3139, 0.3304, 0.3439, 0.3244,
        0.3194, 0.3320, 0.3330, 0.3363, 0.3305, 0.3374, 0.3221, 0.3410,
        0.3271, 0.3450,
    ]
    ax2 = axes[1]
    ax2.plot(range(1, len(bc_val) + 1), bc_val, color="#5DADE2", lw=2)
    ax2.axhline(min(bc_val), color="grey", ls="--", lw=1,
                label=f"Best val MSE = {min(bc_val):.4f}")
    ax2.set_title("BC Validation MSE over Epochs", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Validation MSE")
    ax2.legend(fontsize=9)

    plt.suptitle("BC vs PPO Performance Comparison", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  [OK] model_comparison.png")


# ================================================================
# 4. RADAR CHART  (verified column names)
# ================================================================

def radar_chart():
    """Radar chart of per-scenario policy metrics."""

    df = pd.read_csv(f"{ARTIFACT_DIR}/scenario_summary.csv")
    df.columns = df.columns.str.strip()

    # ── FIXED: use actual column names from Phase-4 output
    metrics = ["success_rate", "collision_rate", "timeout_rate", "off_track_rate"]
    present = [m for m in metrics if m in df.columns]

    if not present:
        print(f"  [SKIP] radar_chart — none of {metrics} found in {df.columns.tolist()}")
        return

    labels  = present
    N       = len(labels)
    values  = df[labels].mean().values
    angles  = np.linspace(0, 2 * np.pi, N, endpoint=False)

    # Close the loop
    values  = np.concatenate((values,  [values[0]]))
    angles  = np.concatenate((angles,  [angles[0]]))

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, values, color="#E74C3C", lw=2)
    ax.fill(angles, values, color="#E74C3C", alpha=0.20)
    ax.set_thetagrids(angles[:-1] * 180 / np.pi,
                      [l.replace("_", "\n") for l in labels], fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_title("Policy Evaluation Radar\n(mean across scenarios)",
                 fontsize=12, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/radar_metrics.png", dpi=150)
    plt.close()
    print("  [OK] radar_metrics.png")


# ================================================================
# 5. TRAINING CURVES  (embedded data — no external CSV required)
# ================================================================

# Raw PPO training log extracted from Phase-3__3_.ipynb
_PPO_LOG_RAW = """
     0     1.467     5.4392   243.7399    0.000    1.479     38.9
    10     1.036     0.4748   142.7745    0.000    1.354     25.4
    20     0.924     0.5150   132.7088    0.000    1.435     21.1
    30     0.826     0.3929   114.5835    0.000    1.445     18.8
    40     0.382     0.3873   100.8528    0.062    1.435     17.4
    50     0.940     0.3973    85.6628    0.031    1.454     16.5
    60     0.681     0.4288    88.9165    0.031    1.373     20.2
    70     0.984     0.4084   111.7485    0.000    1.371     32.1
    80     1.239     0.3802   115.2493    0.000    1.446     39.4
    90     1.133     0.3413   114.6740    0.000    1.538     39.3
   100     0.951     0.3914    97.9247    0.000    1.533     39.3
   200     1.037     0.3130   137.2949    0.031    1.484     39.5
   300     0.893     0.3068   159.3429    0.031    1.558     39.0
   400     0.859     0.3033    78.4884    0.000    0.643     25.6
   500     0.663     0.3018   121.4682    0.000    0.737     29.9
   600     0.714     0.3412   122.4292    0.000    0.843     34.0
   700     0.808     0.2659    95.6036    0.000    0.942     35.7
   800     0.340     0.2673    86.9432    0.031    0.793     24.3
   900     0.725     0.2642   122.5101    0.000    0.697     30.9
  1000     0.813     0.2575   151.5050    0.000    0.768     31.1
  1100     0.657     0.2564   119.3031    0.000    0.779     32.3
  1200     0.857     0.2437   134.4156    0.000    0.689     30.3
  1300     0.965     0.2368   128.7518    0.000    0.726     32.3
  1400     0.761     0.2319   149.8250    0.000    0.622     27.2
  1500     0.811     0.2196   135.8208    0.000    0.769     33.3
  1600     0.407     0.2290   150.6730    0.031    0.881     31.1
  1700    -0.091     0.2274   157.1818    0.094    0.668     28.5
  1800     0.863     0.2208   178.5359    0.000    0.690     28.6
  1900     0.860     0.2284   161.6755    0.000    0.824     32.2
  2000     0.656     0.2161   136.1984    0.000    0.847     35.3
  2100     0.804     0.2175   146.1610    0.000    0.676     29.7
  2200     0.508     0.2276   204.3986    0.031    0.721     28.6
  2300     0.713     0.2211   205.0749    0.000    0.692     24.6
  2400     0.854     0.2133   172.3919    0.000    0.752     27.8
  2500     0.506     0.2204   163.2692    0.000    0.709     24.7
  2600     0.313     0.2137   175.5508    0.031    0.743     29.2
  2700     0.663     0.2258   196.1610    0.031    0.744     28.2
  2800     0.462     0.2292   202.1595    0.031    0.581     28.9
  2900     0.759     0.2223   193.5765    0.000    0.704     26.9
  3000     0.854     0.2179   180.8129    0.000    0.813     28.0
  3100     0.820     0.2164   187.1062    0.000    0.847     26.8
  3200     0.447     0.2371   183.9329    0.000    0.717     28.5
  3300     0.717     0.2319   225.5604    0.000    0.671     24.9
  3400     0.857     0.2171   183.9233    0.000    0.806     25.5
  3500     0.859     0.2248   182.4919    0.000    0.784     25.4
  3600     0.660     0.2252   192.3647    0.000    0.876     28.4
  3700     0.794     0.2173   181.4304    0.000    0.775     25.3
  3800     0.912     0.2272   202.9313    0.000    0.855     29.1
  3900     0.807     0.2181   165.4532    0.000    0.733     23.3
  4000     0.805     0.2296   225.5827    0.000    0.756     23.8
  4100     0.808     0.2179   179.9678    0.000    0.782     24.9
  4200     0.704     0.2113   185.8580    0.031    0.793     23.4
  4300     0.454     0.2183   202.3895    0.031    0.801     26.9
  4400     0.593     0.2109   170.9844    0.000    0.869     28.7
  4500     0.905     0.2130   163.8027    0.000    0.811     26.3
  4600     0.952     0.2299   215.3019    0.000    0.698     24.6
  4700     0.704     0.2195   213.9388    0.000    0.687     27.6
  4800     0.559     0.2216   199.7210    0.031    0.820     26.8
  4900     0.978     0.3421   223.6576    0.000    0.717     24.0
  5000     0.810     0.2080   249.4876    0.000    0.746     22.5
  5100     1.004     0.2240   234.4883    0.000    0.665     25.8
  5200     0.752     0.2113   197.8042    0.000    0.800     26.0
  5300     0.368     0.2415   214.0791    0.031    0.748     19.0
  5400     0.957     0.2422   203.5548    0.000    0.763     25.6
  5500     0.859     0.2166   188.4469    0.000    0.814     26.0
  5600     0.603     0.2070   202.3761    0.031    0.689     27.7
  5700     0.556     0.2469   230.0426    0.031    0.795     26.2
  5800     0.947     0.2268   249.8809    0.000    0.685     19.4
  5900     0.812     0.2131   226.7812    0.000    0.779     22.4
  5990     0.553     0.5540   274.0476    0.031    0.699     22.7
"""

# Curriculum advancement thresholds (from notebook)
_CURRICULUM_ITERS = [300, 700, 1000, 1300, 1600]
_CURRICULUM_LABELS = ["0%→10%", "10%→25%", "25%→50%", "50%→75%", "75%→100%"]


def _parse_ppo_log():
    rows = []
    for line in _PPO_LOG_RAW.strip().splitlines():
        parts = line.split()
        if len(parts) >= 7:
            try:
                it  = int(parts[0])
                rew = float(parts[1])
                prog = float(parts[6])
                rows.append({"iter": it, "reward": rew, "progress_m": prog})
            except ValueError:
                pass
    return pd.DataFrame(rows)


def plot_training_curves():
    """
    Dual-axis training curves: reward (left) and mean progress in metres (right).
    Vertical lines mark curriculum level advances.
    """

    df = _parse_ppo_log()

    fig, ax1 = plt.subplots(figsize=(12, 5))

    color_rew  = "#2980B9"
    color_prog = "#E74C3C"

    ax1.plot(df["iter"], df["reward"], color=color_rew, lw=1.8, label="Reward")
    ax1.set_xlabel("PPO Iteration", fontsize=11)
    ax1.set_ylabel("Mean Episode Reward", color=color_rew, fontsize=11)
    ax1.tick_params(axis="y", labelcolor=color_rew)

    ax2 = ax1.twinx()
    ax2.plot(df["iter"], df["progress_m"], color=color_prog,
             lw=1.8, ls="--", label="Progress (m)")
    ax2.set_ylabel("Mean Progress (m)", color=color_prog, fontsize=11)
    ax2.tick_params(axis="y", labelcolor=color_prog)

    # Curriculum lines
    for it, lbl in zip(_CURRICULUM_ITERS, _CURRICULUM_LABELS):
        ax1.axvline(it, color="grey", ls=":", lw=1.2, alpha=0.8)
        ax1.text(it + 30, ax1.get_ylim()[0] + 0.05, lbl,
                 rotation=90, fontsize=7, color="grey", va="bottom")

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               loc="upper right", fontsize=9)

    ax1.set_title("PPO Training Curves (5 Curriculum Levels, 5,990 Iterations)",
                  fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/training_reward.png", dpi=150)
    plt.close()
    print("  [OK] training_reward.png")


# ================================================================
# RUN ALL
# ================================================================

def main():
    print("Generating figures...")
    plot_architecture()
    plot_evaluation_table()
    compare_models()
    radar_chart()
    plot_training_curves()
    print(f"\nAll figures saved to: {FIG_DIR}")


if __name__ == "__main__":
    main()

Generating figures...
  [OK] system_architecture.png
  [OK] progress_by_scenario.png
  [OK] model_comparison.png
  [OK] radar_metrics.png
  [OK] training_reward.png

All figures saved to: artifacts/phase4/figures
